In [1]:
import pandas as pd
DATA_PATH = 'jobpostingdata.csv'
MODEL_PATH = 'model.pkl'

df = pd.read_csv(DATA_PATH)

In [3]:
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [4]:
def get_tag(tag):
    if tag[0] in ['V', 'N', 'R']:
        return tag[0].lower()
    elif tag[0] == 'J':
        return 'a'
    else:
        return 'n'

def lemmatization(tokens):
    lemmatizer = WordNetLemmatizer()
    tagged = pos_tag(tokens)
    lemmatized = []
    for word, tag in tagged:
        result = lemmatizer.lemmatize(word, get_tag(tag))
        lemmatized.append(result)
    return lemmatized

def text_preprocessor(sentence):
    eng_stopwords = stopwords.words('eng')
    tokens = word_tokenize(sentence)
    cleaned = [
        token.lower() for token in tokens if token not in eng_stopwords
        and
        token.isalpha()
    ]
    lemmatized = lemmatization(cleaned)
    return lemmatized

In [5]:
def feature_extraction(sentence, existed_words):
    unique_words = set(text_preprocessor(sentence))
    feature = {}
    for word in existed_words:
        feature[word] = (word in unique_words)
    return feature

In [7]:
from nltk.classify.naivebayes import NaiveBayesClassifier
from nltk.classify import accuracy
from sklearn.model_selection import train_test_split

In [6]:
def train_classifier():
    x = df['text']
    y = df['fraudulent']

    corpus = ' '.join(x)
    existed_words = set(text_preprocessor(corpus))

    features = [
        (feature_extraction(sentence, existed_words), category)
        for sentence, category in zip(x, y)
    ]

    train_data, test_data = train_test_split(
        features, test_size=0.2, random_state=42
    )
    classifier = NaiveBayesClassifier.train(train_data)
    acc = accuracy(classifier, test_data)
    print(acc)

    return classifier, existed_words

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer 

In [8]:
def train_vectorizer():
    x = df['text']
    vectorizer = TfidfVectorizer(
        ngram_range=(1,2),
        stop_words='english'
    )
    vectors = vectorizer.fit_transform(x)
    
    return vectorizer, vectors

In [9]:
import os
import pickle

In [11]:
def load_model():
    if os.path.exists(MODEL_PATH):
        with open(MODEL_PATH, 'rb') as f:
            classifier, existed_words, vectorizer, vectors = pickle.load(f)
        return classifier, existed_words, vectorizer, vectors
    else:
        classifier, existed_words = train_classifier()
        vectorizer, vectors = train_vectorizer()
        with open(MODEL_PATH, 'wb') as f:
            pickle.dump((classifier, existed_words, vectorizer, vectors), f)
        return classifier, existed_words, vectorizer, vectors

In [12]:
def print_menu(sentence, category):
    print(sentence)
    print(category)
    print("1. Write")
    print("2. Recommend")
    print("3. NER")
    print("4. Exit")

    return input("Choice: ")

In [13]:
def write_sentence():
    sent = ''
    while len(sent.strip()) < 20 or len(sent.strip().split()) < 3:
        sent = input("Sent: ")
    return sent

In [18]:
def classify_text(classifier, sentence, existed_words):
    feature = feature_extraction(sentence, existed_words)
    category = classifier.classify(feature)
    if category == 1:
        return 'TRUE'
    else:
        return 'FALSE'

In [14]:
def load_nlp(nlp, sentence):
    doc = nlp(sentence)
    ents_dicts = {}

    for ent in doc.ents:
        if ent.label_ not in ents_dicts.keys():
            ents_dicts[ent.label_] = []
        ents_dicts[ent.label_].append(doc.text)
    return ents_dicts

In [15]:
def print_NER(ents_dicts):
    for key, value in ents_dicts.items():
        print(f"{key}: {value}")

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
def recommend_top_n(vectorizer: TfidfVectorizer, job_vectors, query, topn=5):
    vectorized_query = vectorizer.transform([query])
    similarity = cosine_similarity(job_vectors, vectorized_query).flatten()
    topidx = similarity.argsort()[::-1][:topn]

    for i, idx in enumerate(topidx, 1):
        print(f"{i}. {df['title'].iloc[idx]}")

In [19]:
import spacy

In [22]:
def menu():
    sent = ''
    cat = ''

    classifier = None
    existed_words = None
    vectorizer = None
    vectors = None

    nlp = spacy.load("en_core_web_sm")
    ents_dicts = {}

    while True:
        choice = print_menu(sent, cat)

        if choice == '1':
            sent = write_sentence()
            if classifier is None and existed_words is None or vectorizer is None or vectors is None:
                classifier, existed_words, vectorizer, vectors = load_model()
        
        elif choice == '2':
            recommend_top_n(vectorizer, vectors, sent)
        
        elif choice == '3':
            ents_dicts = load_nlp(nlp, sent)
            print_NER(ents_dicts)



In [ ]:
menu()



1. Write
2. Recommend
3. NER
4. Exit


c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.3.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.3.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


I have a job tomorrow at cafeteria

1. Write
2. Recommend
3. NER
4. Exit
1. Part Time Job Work From Home, Daily Pay.
2. Frac Supervisor
3. Electrical Maintenance Technician
4. Well Test Operator
5. Injection Molding Supervisor
I have a job tomorrow at cafeteria

1. Write
2. Recommend
3. NER
4. Exit
DATE: ['I have a job tomorrow at cafeteria']
I have a job tomorrow at cafeteria

1. Write
2. Recommend
3. NER
4. Exit
DATE: ['I have a job tomorrow at cafeteria']
I have a job tomorrow at cafeteria

1. Write
2. Recommend
3. NER
4. Exit
I have a job tomorrow at cafeteria

1. Write
2. Recommend
3. NER
4. Exit
